# First Clean — PREPARE_GDB

This notebook cleans feature classes named `A<BRANCH_CODE>_<ZONE>` in `PREPARE_GDB`.

Approved order:

```text
1. APID + ACQUIRED
2. Check / Repair Geometry
3. INSIDE_X + INSIDE_Y
4. Duplicate cleanup
5. CHANGWAT / 2 km inside-point validation
6. Excel report
```

Key rules:

- `CHANGWAT_CODE` is the first two digits of `BRANCH_CODE` from the feature-class name.
- `_47` must be WKID 24047; `_48` must be WKID 24048.
- `APID = <LANDOFFICE>-<UTM1><UTM2>-<UTM3>-<UTMSCALE>-<UTM4>-<LAND_NO>`.
- Invalid/missing APID components produce `APID = NULL`; they are reported, not deleted.
- `RepairGeometry` uses `KEEP_NULL`; null geometry is retained and reported.
- `INSIDE_X/Y` uses ArcGIS Central Point (`INSIDE_X`, `INSIDE_Y`).
- Automatic duplicate = non-null APID + identical Shape. Survivor = newest valid `LAST_UPD_D`, then highest OBJECTID.
- Identical Shape with `APID = NULL` is report-only.
- Boundary deletion uses the stored `INSIDE_X/Y` point. Keep if within 2,000 m of the correct CHANGWAT polygon; delete if farther than 2,000 m.
- If the CHANGWAT reference cannot be resolved, skip boundary deletion for that feature class.
- The workflow is resumable. Destructive duplicate/boundary OID plans are persisted before deletion.

Official ArcGIS references used by this notebook:

- Calculate Geometry Attributes: `https://pro.arcgis.com/en/pro-app/latest/tool-reference/data-management/calculate-geometry-attributes.htm`
- Check Geometry / Repair Geometry: `https://pro.arcgis.com/en/pro-app/latest/tool-reference/data-management/check-geometry.htm`
- Find Identical: `https://pro.arcgis.com/en/pro-app/latest/tool-reference/data-management/find-identical.htm`
- Select Layer By Location: `https://pro.arcgis.com/en/pro-app/latest/tool-reference/data-management/select-layer-by-location.htm`

In [1]:
from pathlib import Path
from collections import Counter, defaultdict
import datetime as dt
import json, os, re, traceback, uuid

try:
    import arcpy
except ImportError as exc:
    raise RuntimeError('Run with the ArcGIS Pro Python environment (arcpy required).') from exc

arcpy.env.overwriteOutput = True

# ------------------------------------------------------------------
# Configuration
# ------------------------------------------------------------------
ACQUIRED = '2022-05'
BASE_DIR = Path(r'D:\A02-Projects\Refine')
PREPARE_GDB = BASE_DIR / '01_acquired' / f'A{ACQUIRED}.gdb'
REPORT_DIR = BASE_DIR / '05_reports' / ACQUIRED
ACQUIRED_REPORT = REPORT_DIR / f'02_{ACQUIRED}-first_Clean.xlsx'
BRANCH_LOOKUP = Path(r'D:\A03-Resources\Infomation\API\IPADDRESS.xlsx')
BRANCH_LOOKUP_SHEET = 'OldBranch'  # retained for project consistency; not used here
REFERENCE_GDB = BASE_DIR / '02_ref' / 'district.gdb'
CONTROL_GDB = REPORT_DIR / f'_clean_control_{ACQUIRED.replace("-", "_")}.gdb'
SCRATCH_GDB = REPORT_DIR / f'_clean_scratch_{ACQUIRED.replace("-", "_")}.gdb'

ZONE_WKID = {'47': 24047, '48': 24048}
BOUNDARY_DISTANCE = '2000 Meters'
DUPLICATE_XY_TOLERANCE = '0 Meters'
WRITE_REPORT_AFTER_EACH_FC = True
STOP_ON_ERROR = False
KEEP_SCRATCH_GDB = False
POINT_PROGRESS_EVERY = 250_000
FC_RE = re.compile(r'^A(?P<branch>\d{8})_(?P<zone>47|48)$', re.I)
APID_FIELDS = ['LANDOFFICE','UTM1','UTM2','UTM3','UTMSCALE','UTM4','LAND_NO']
UTM_SCALES = {500, 1000, 2000, 4000, 5000}

print('ACQUIRED      :', ACQUIRED)
print('PREPARE_GDB  :', PREPARE_GDB)
print('REFERENCE_GDB:', REFERENCE_GDB)
print('REPORT       :', ACQUIRED_REPORT)
print('CONTROL_GDB  :', CONTROL_GDB)


ACQUIRED      : 2022-05
PREPARE_GDB  : D:\A02-Projects\Refine\01_acquired\A2022-05.gdb
REFERENCE_GDB: D:\A02-Projects\Refine\02_ref\district.gdb
REPORT       : D:\A02-Projects\Refine\05_reports\2022-05\02_2022-05-first_Clean.xlsx
CONTROL_GDB  : D:\A02-Projects\Refine\05_reports\2022-05\_clean_control_2022_05.gdb


## Core helpers and persistent state

Technical state/report data is stored in `_clean_control_<ACQUIRED>.gdb`. `PREPARE_GDB` is never rebuilt by this notebook.

In [2]:
def now_text(): return dt.datetime.now().strftime('%Y-%m-%d %H:%M:%S')

def norm(v):
    if v is None: return None
    if isinstance(v, float) and v.is_integer(): v = int(v)
    s = str(v).strip()
    return s or None

def as_long(v):
    if v is None: return None
    if isinstance(v, int): return v
    if isinstance(v, float):
        if v.is_integer(): return int(v)
        raise ValueError(v)
    s = norm(v)
    if s is None: return None
    n = float(s)
    if not n.is_integer(): raise ValueError(v)
    return int(n)

def valid_digits(v, n):
    s = norm(v)
    return bool(s and re.fullmatch(rf'\d{{{n}}}', s))

def valid_last_upd(v):
    s = norm(v)
    if not s or not re.fullmatch(r'\d{14}', s): return None
    try: return dt.datetime.strptime(s, '%Y%m%d%H%M%S')
    except ValueError: return None

def fmap(ds): return {f.name.upper(): f for f in arcpy.ListFields(str(ds))}
def field(ds, name): return fmap(ds).get(name.upper())
def nrows(ds): return int(arcpy.management.GetCount(str(ds))[0])

def wkid(ds):
    sr = arcpy.Describe(str(ds)).spatialReference
    return int(getattr(sr, 'factoryCode', 0) or getattr(sr, 'PCSCode', 0) or 0)

def ensure_field(ds, name, typ, length=None):
    f = field(ds, name)
    expected = {'TEXT':'String','DOUBLE':'Double','LONG':'Integer','SHORT':'SmallInteger'}[typ]
    if f:
        if f.type != expected:
            raise RuntimeError(f'{Path(ds).name}.{name}: {f.type}, expected {expected}')
        return
    kw = {'field_length': int(length)} if typ == 'TEXT' else {}
    arcpy.management.AddField(str(ds), name, typ, **kw)

def sql_eq(ds, name, value):
    f = field(ds, name)
    if not f: raise RuntimeError(f'Missing field {name}: {ds}')
    q = arcpy.AddFieldDelimiters(str(ds), f.name)
    if f.type == 'String': return f"{q} = '{str(value).replace(chr(39), chr(39)*2)}'"
    return f'{q} = {value}'

def sql_null(ds, name, is_null=True):
    f = field(ds, name)
    if not f: raise RuntimeError(f'Missing field {name}: {ds}')
    q = arcpy.AddFieldDelimiters(str(ds), f.name)
    return f'{q} IS ' + ('' if is_null else 'NOT ') + 'NULL'

def create_gdb(path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    arcpy.management.CreateFileGDB(str(path.parent), path.name)

def setup():
    if not arcpy.Exists(str(PREPARE_GDB)): raise FileNotFoundError(PREPARE_GDB)
    if not arcpy.Exists(str(REFERENCE_GDB)): raise FileNotFoundError(REFERENCE_GDB)
    REPORT_DIR.mkdir(parents=True, exist_ok=True)
    if not arcpy.Exists(str(CONTROL_GDB)): create_gdb(CONTROL_GDB)
    if arcpy.Exists(str(SCRATCH_GDB)): arcpy.management.Delete(str(SCRATCH_GDB))
    create_gdb(SCRATCH_GDB)

def list_fcs():
    old = arcpy.env.workspace
    try:
        arcpy.env.workspace = str(PREPARE_GDB)
        out = []
        for name in arcpy.ListFeatureClasses() or []:
            m = FC_RE.fullmatch(name)
            if m: out.append((name, m.group('branch'), m.group('zone')))
        return sorted(out)
    finally: arcpy.env.workspace = old

def identity(fc_name, branch, zone):
    fc = PREPARE_GDB / fc_name
    actual = wkid(fc); expected = ZONE_WKID[zone]
    if actual != expected: raise RuntimeError(f'{fc_name}: WKID {actual}, expected {expected}')
    if str(arcpy.Describe(str(fc)).shapeType).lower() != 'polygon':
        raise RuntimeError(f'{fc_name}: not Polygon')
    return {'FEATURE_CLASS':fc_name,'BRANCH_CODE':branch,'CHANGWAT_CODE':branch[:2],'ZONE':zone,'WKID':actual}

STATE = CONTROL_GDB / 'CLEAN_STATE'
DUP_PLAN = CONTROL_GDB / 'DUP_DELETE_PLAN'
NULL_REVIEW = CONTROL_GDB / 'NULL_APID_REVIEW'
BOUNDARY_PLAN = CONTROL_GDB / 'BOUNDARY_DELETE_PLAN'
ISSUES = CONTROL_GDB / 'ISSUES'
R1 = CONTROL_GDB / 'PROCESS_1'
RG = CONTROL_GDB / 'GEOMETRY'
RD = CONTROL_GDB / 'DUPLICATES'
RB = CONTROL_GDB / 'BOUNDARY'
RFC = CONTROL_GDB / 'FEATURE_CLASSES'
RS = CONTROL_GDB / 'SUMMARY'

STATE_SCHEMA=[('FC_NAME','TEXT',100),('STATUS','TEXT',30),('START_COUNT','LONG',None),('PROCESS1_DONE','SHORT',None),('GEOMETRY_DONE','SHORT',None),('INSIDE_DONE','SHORT',None),('DUP_PLAN_READY','SHORT',None),('DUPLICATES_DONE','SHORT',None),('BOUNDARY_PLAN_READY','SHORT',None),('BOUNDARY_DONE','SHORT',None),('FINAL_COUNT','LONG',None),('LAST_ERROR','TEXT',2000),('UPDATED_AT','TEXT',19)]
DUP_SCHEMA=[('FC_NAME','TEXT',100),('DELETE_OID','LONG',None),('KEEP_OID','LONG',None),('FEAT_SEQ','LONG',None),('DELETE_LAST_UPD','TEXT',50),('KEEP_LAST_UPD','TEXT',50)]
NULL_SCHEMA=[('FC_NAME','TEXT',100),('FEATURE_OID','LONG',None),('FEAT_SEQ','LONG',None)]
BPLAN_SCHEMA=[('FC_NAME','TEXT',100),('DELETE_OID','LONG',None)]
ISSUE_SCHEMA=[('FC_NAME','TEXT',100),('PROCESS','TEXT',40),('ISSUE_CODE','TEXT',100),('MESSAGE','TEXT',2000),('ISSUE_COUNT','LONG',None)]
R1_SCHEMA=[('FC_NAME','TEXT',100),('BRANCH_CODE','TEXT',8),('CHANGWAT_CODE','TEXT',2),('ZONE','TEXT',2),('COUNT_INPUT','LONG',None),('APID_VALID','LONG',None),('APID_NULL','LONG',None),('ACQUIRED_COUNT','LONG',None),('BRANCH_ATTR_MISMATCH','LONG',None),('CHANGWAT_ATTR_MISMATCH','LONG',None),('MISSING_APID_FIELDS','TEXT',1000),('APID_ISSUE_BREAKDOWN','TEXT',2000)]
RG_SCHEMA=[('FC_NAME','TEXT',100),('COUNT_BEFORE','LONG',None),('PROBLEM_ROWS_BEFORE','LONG',None),('PROBLEM_FEATURES_BEFORE','LONG',None),('PROBLEM_TYPES_BEFORE','TEXT',2000),('COUNT_AFTER','LONG',None),('PROBLEM_ROWS_AFTER','LONG',None),('PROBLEM_FEATURES_AFTER','LONG',None),('PROBLEM_TYPES_AFTER','TEXT',2000)]
RD_SCHEMA=[('FC_NAME','TEXT',100),('COUNT_BEFORE','LONG',None),('DUP_GROUPS','LONG',None),('DUP_ROWS','LONG',None),('DUP_DELETE_PLANNED','LONG',None),('DUP_DELETED','LONG',None),('NULL_APID_IDENTICAL_GROUPS','LONG',None),('NULL_APID_IDENTICAL_ROWS','LONG',None),('COUNT_AFTER','LONG',None)]
RB_SCHEMA=[('FC_NAME','TEXT',100),('CHANGWAT_CODE','TEXT',2),('ZONE','TEXT',2),('REFERENCE_FC','TEXT',100),('REFERENCE_MATCH_COUNT','LONG',None),('COUNT_BEFORE','LONG',None),('INSIDE_POINT_COUNT','LONG',None),('INSIDE_POINT_NULL','LONG',None),('KEEP_WITHIN_2KM','LONG',None),('DELETE_PLANNED','LONG',None),('DELETE_COUNT','LONG',None),('COUNT_AFTER','LONG',None),('STATUS','TEXT',40),('SKIP_REASON','TEXT',2000)]
RFC_SCHEMA=[('FC_NAME','TEXT',100),('BRANCH_CODE','TEXT',8),('CHANGWAT_CODE','TEXT',2),('ZONE','TEXT',2),('WKID','LONG',None),('COUNT_START','LONG',None),('APID_NULL','LONG',None),('GEOM_PROBLEM_FEATURES','LONG',None),('DUP_DELETED','LONG',None),('BOUNDARY_DELETED','LONG',None),('COUNT_FINAL','LONG',None),('STATUS','TEXT',40)]
RS_SCHEMA=[('METRIC','TEXT',100),('VALUE','LONG',None)]

def ensure_table(path, schema):
    if arcpy.Exists(str(path)): return
    arcpy.management.CreateTable(str(CONTROL_GDB), Path(path).name)
    for name, typ, length in schema:
        kw={'field_length':int(length)} if typ=='TEXT' else {}
        arcpy.management.AddField(str(path), name, typ, **kw)

def ensure_control():
    for p,s in [(STATE,STATE_SCHEMA),(DUP_PLAN,DUP_SCHEMA),(NULL_REVIEW,NULL_SCHEMA),(BOUNDARY_PLAN,BPLAN_SCHEMA),(ISSUES,ISSUE_SCHEMA),(R1,R1_SCHEMA),(RG,RG_SCHEMA),(RD,RD_SCHEMA),(RB,RB_SCHEMA),(RFC,RFC_SCHEMA),(RS,RS_SCHEMA)]: ensure_table(p,s)

def delete_where(table, where=None):
    with arcpy.da.UpdateCursor(str(table), ['OID@'], where) as c:
        for _ in c: c.deleteRow()

def delete_fc_rows(table, fc): delete_where(table, sql_eq(table,'FC_NAME',fc))

def state(fc):
    names=[x[0] for x in STATE_SCHEMA]; where=sql_eq(STATE,'FC_NAME',fc)
    with arcpy.da.SearchCursor(str(STATE), names, where) as c:
        for row in c: return dict(zip(names,row))
    return None

def ensure_state(fc, start_count):
    s=state(fc)
    if s: return s
    rec={'FC_NAME':fc,'STATUS':'PENDING','START_COUNT':start_count,'PROCESS1_DONE':0,'GEOMETRY_DONE':0,'INSIDE_DONE':0,'DUP_PLAN_READY':0,'DUPLICATES_DONE':0,'BOUNDARY_PLAN_READY':0,'BOUNDARY_DONE':0,'FINAL_COUNT':None,'LAST_ERROR':None,'UPDATED_AT':now_text()}
    names=[x[0] for x in STATE_SCHEMA]
    with arcpy.da.InsertCursor(str(STATE),names) as c: c.insertRow([rec[n] for n in names])
    return state(fc)

def set_state(fc, **kw):
    kw.setdefault('UPDATED_AT',now_text()); names=list(kw)
    with arcpy.da.UpdateCursor(str(STATE),names,sql_eq(STATE,'FC_NAME',fc)) as c:
        for row in c:
            for i,n in enumerate(names): row[i]=kw[n]
            c.updateRow(row); return
    raise RuntimeError(f'Missing state: {fc}')

def done(fc, flag):
    s=state(fc); return bool(s and int(s.get(flag) or 0)==1)

def upsert(table, schema, fc, rec):
    names=[x[0] for x in schema]; where=sql_eq(table,'FC_NAME',fc); found=False
    with arcpy.da.UpdateCursor(str(table),names,where) as c:
        for row in c:
            for i,n in enumerate(names): row[i]=rec.get(n)
            c.updateRow(row); found=True
    if not found:
        with arcpy.da.InsertCursor(str(table),names) as c: c.insertRow([rec.get(n) for n in names])

def report_row(table,schema,fc):
    names=[x[0] for x in schema]
    with arcpy.da.SearchCursor(str(table),names,sql_eq(table,'FC_NAME',fc)) as c:
        for row in c: return dict(zip(names,row))
    return None

def clear_issues(fc, process):
    w=f"({sql_eq(ISSUES,'FC_NAME',fc)}) AND ({sql_eq(ISSUES,'PROCESS',process)})"
    delete_where(ISSUES,w)

def issue(fc,process,code_,message,count=1):
    with arcpy.da.InsertCursor(str(ISSUES),['FC_NAME','PROCESS','ISSUE_CODE','MESSAGE','ISSUE_COUNT']) as c:
        c.insertRow([fc,process,code_,message,int(count)])


## Process 1 — APID and ACQUIRED

All APID components are validated. Missing/invalid components leave APID NULL and are counted by reason.

In [3]:
def valid_utm4(scale, v):
    s=norm(v)
    if scale is None or s is None: return False
    if scale==5000: return bool(re.fullmatch(r'\d{3}',s))
    if scale==4000: return s=='00'
    if scale==2000: return s in {'01','02','03','04'}
    if scale==1000: return bool(re.fullmatch(r'\d{2}',s) and 1<=int(s)<=16)
    if scale==500: return bool(re.fullmatch(r'\d{2}',s) and 1<=int(s)<=64)
    return False

def make_apid(v):
    try: office=as_long(v.get('LANDOFFICE'))
    except: return None,'INVALID_LANDOFFICE'
    u1=norm(v.get('UTM1')); u3=norm(v.get('UTM3')); u4=norm(v.get('UTM4')); land=norm(v.get('LAND_NO'))
    try: u2=as_long(v.get('UTM2'))
    except: return None,'INVALID_UTM2'
    try: scale=as_long(v.get('UTMSCALE'))
    except: return None,'INVALID_UTMSCALE'
    if office is None: return None,'MISSING_LANDOFFICE'
    if not valid_digits(u1,4): return None,'INVALID_UTM1'
    if u2 not in {1,2,3,4}: return None,'INVALID_UTM2'
    if not valid_digits(u3,4): return None,'INVALID_UTM3'
    if scale not in UTM_SCALES: return None,'INVALID_UTMSCALE'
    if scale==5000 and u3!='0000': return None,'UTM3_SCALE_CONFLICT'
    if not valid_utm4(scale,u4): return None,'UTM4_SCALE_CONFLICT'
    if land is None: return None,'MISSING_LAND_NO'
    return f'{office}-{u1}{u2}-{u3}-{scale}-{u4}-{land}',None

def process1(fc_name, ident):
    if done(fc_name,'PROCESS1_DONE'): print('  Process 1: already complete'); return
    print('  Process 1: APID + ACQUIRED'); clear_issues(fc_name,'PROCESS_1')
    fc=PREPARE_GDB/fc_name; ensure_field(fc,'APID','TEXT',100); ensure_field(fc,'ACQUIRED','TEXT',10)
    fm=fmap(fc); missing=[n for n in APID_FIELDS if n not in fm]
    if missing: issue(fc_name,'PROCESS_1','MISSING_APID_FIELDS','Missing: '+', '.join(missing))
    cf=['OID@','APID','ACQUIRED']; available=[]
    for n in APID_FIELDS:
        if n in fm: cf.append(fm[n].name); available.append(n)
    b=fm.get('BRANCH_CODE'); ch=fm.get('CHANGWAT_CODE')
    if b: cf.append(b.name)
    if ch: cf.append(ch.name)
    ix={n.upper():i for i,n in enumerate(cf) if n!='OID@'}
    valid=null=acq=bmis=cmis=0; counts=Counter()
    with arcpy.da.UpdateCursor(str(fc),cf) as cur:
        for row in cur:
            vals={n:row[ix[n]] for n in available}
            apid,why=(None,'MISSING_APID_FIELDS') if missing else make_apid(vals)
            row[ix['APID']]=apid; row[ix['ACQUIRED']]=ACQUIRED; acq+=1
            if apid: valid+=1
            else: null+=1; counts[why or 'UNKNOWN']+=1
            if b and norm(row[ix[b.name.upper()]]) not in (None,ident['BRANCH_CODE']): bmis+=1
            if ch and norm(row[ix[ch.name.upper()]]) not in (None,ident['CHANGWAT_CODE']): cmis+=1
            cur.updateRow(row)
    if bmis: issue(fc_name,'PROCESS_1','BRANCH_CODE_NAME_CONFLICT','BRANCH_CODE conflicts with FC name.',bmis)
    if cmis: issue(fc_name,'PROCESS_1','CHANGWAT_CODE_NAME_CONFLICT','CHANGWAT_CODE conflicts with FC name.',cmis)
    for k,v in counts.items(): issue(fc_name,'PROCESS_1',k,f'{v:,} row(s): {k}',v)
    rec={'FC_NAME':fc_name,'BRANCH_CODE':ident['BRANCH_CODE'],'CHANGWAT_CODE':ident['CHANGWAT_CODE'],'ZONE':ident['ZONE'],'COUNT_INPUT':nrows(fc),'APID_VALID':valid,'APID_NULL':null,'ACQUIRED_COUNT':acq,'BRANCH_ATTR_MISMATCH':bmis,'CHANGWAT_ATTR_MISMATCH':cmis,'MISSING_APID_FIELDS':', '.join(missing),'APID_ISSUE_BREAKDOWN':json.dumps(dict(counts),ensure_ascii=False,sort_keys=True)}
    upsert(R1,R1_SCHEMA,fc_name,rec); set_state(fc_name,PROCESS1_DONE=1,STATUS='PROCESSING',LAST_ERROR=None)


## Process 2 — Check / Repair Geometry

Geometry problems are counted before and after repair. `KEEP_NULL` prevents silent deletion of null geometries.

In [4]:
def geom_summary(fc,out_table):
    if arcpy.Exists(str(out_table)): arcpy.management.Delete(str(out_table))
    arcpy.management.CheckGeometry(str(fc),str(out_table),'ESRI')
    rows=0; fids=set(); types=Counter()
    with arcpy.da.SearchCursor(str(out_table),['FEATURE_ID','PROBLEM']) as c:
        for fid,problem in c:
            rows+=1
            if fid is not None: fids.add(int(fid))
            types[norm(problem) or '<NULL>']+=1
    return rows,len(fids),dict(types)

def process2_geometry(fc_name):
    if done(fc_name,'GEOMETRY_DONE'): print('  Process 2: already complete'); return
    print('  Process 2: Check + Repair Geometry'); clear_issues(fc_name,'GEOMETRY')
    fc=PREPARE_GDB/fc_name; existing=report_row(RG,RG_SCHEMA,fc_name)
    before_tbl=SCRATCH_GDB/f'GB_{uuid.uuid4().hex[:8]}'; after_tbl=SCRATCH_GDB/f'GA_{uuid.uuid4().hex[:8]}'
    if existing is None:
        br,bf,bt=geom_summary(fc,before_tbl)
        rec={'FC_NAME':fc_name,'COUNT_BEFORE':nrows(fc),'PROBLEM_ROWS_BEFORE':br,'PROBLEM_FEATURES_BEFORE':bf,'PROBLEM_TYPES_BEFORE':json.dumps(bt,ensure_ascii=False,sort_keys=True),'COUNT_AFTER':None,'PROBLEM_ROWS_AFTER':None,'PROBLEM_FEATURES_AFTER':None,'PROBLEM_TYPES_AFTER':None}
        upsert(RG,RG_SCHEMA,fc_name,rec)
    else: rec=existing
    arcpy.management.RepairGeometry(str(fc),'KEEP_NULL','ESRI')
    ar,af,at=geom_summary(fc,after_tbl)
    rec.update({'COUNT_AFTER':nrows(fc),'PROBLEM_ROWS_AFTER':ar,'PROBLEM_FEATURES_AFTER':af,'PROBLEM_TYPES_AFTER':json.dumps(at,ensure_ascii=False,sort_keys=True)})
    upsert(RG,RG_SCHEMA,fc_name,rec)
    if af: issue(fc_name,'GEOMETRY','GEOMETRY_PROBLEMS_REMAIN','Geometry problems remain after repair.',af)
    if int(rec['COUNT_BEFORE'])!=int(rec['COUNT_AFTER']): issue(fc_name,'GEOMETRY','GEOMETRY_ROW_COUNT_CHANGED','Row count changed although KEEP_NULL was used.',abs(int(rec['COUNT_BEFORE'])-int(rec['COUNT_AFTER'])))
    set_state(fc_name,GEOMETRY_DONE=1,STATUS='PROCESSING',LAST_ERROR=None)


## Process 3 — INSIDE_X / INSIDE_Y

ArcGIS `CalculateGeometryAttributes` calculates a central point inside/on each polygon. These stored coordinates are later used for the province-distance test.

In [5]:
def process3_inside(fc_name):
    if done(fc_name,'INSIDE_DONE'): print('  Process 3: already complete'); return
    print('  Process 3: INSIDE_X + INSIDE_Y'); clear_issues(fc_name,'INSIDE')
    fc=PREPARE_GDB/fc_name; ensure_field(fc,'INSIDE_X','DOUBLE'); ensure_field(fc,'INSIDE_Y','DOUBLE')
    sr=arcpy.Describe(str(fc)).spatialReference
    arcpy.management.CalculateGeometryAttributes(str(fc),[['INSIDE_X','INSIDE_X'],['INSIDE_Y','INSIDE_Y']],coordinate_system=sr)
    nulls=sum(1 for x,y in arcpy.da.SearchCursor(str(fc),['INSIDE_X','INSIDE_Y']) if x is None or y is None)
    if nulls: issue(fc_name,'INSIDE','INSIDE_XY_NULL','INSIDE_X/Y is NULL after calculation.',nulls)
    set_state(fc_name,INSIDE_DONE=1,STATUS='PROCESSING',LAST_ERROR=None)


## Process 4 — Duplicate cleanup

A persistent delete plan is created before deletion. If execution stops mid-delete, rerun the notebook; it will finish the same OID plan.

In [6]:
def find_identical(layer,out_table,fields_):
    if arcpy.Exists(str(out_table)): arcpy.management.Delete(str(out_table))
    arcpy.management.FindIdentical(layer,str(out_table),fields_,xy_tolerance=DUPLICATE_XY_TOLERANCE,output_record_option='ONLY_DUPLICATES')

def identical_groups(table):
    g=defaultdict(list)
    if nrows(table)==0: return g
    with arcpy.da.SearchCursor(str(table),['IN_FID','FEAT_SEQ']) as c:
        for fid,seq in c: g[int(seq)].append(int(fid))
    return g

def survivor_key(oid,last):
    parsed=valid_last_upd(last)
    return (1,parsed,int(oid)) if parsed else (0,dt.datetime.min,int(oid))

def build_dup_plan(fc_name):
    fc=PREPARE_GDB/fc_name; delete_fc_rows(DUP_PLAN,fc_name); delete_fc_rows(NULL_REVIEW,fc_name); clear_issues(fc_name,'DUPLICATES')
    if not field(fc,'APID'): raise RuntimeError(f'{fc_name}: APID missing')
    last_field=field(fc,'LAST_UPD_D')

    # APID NOT NULL: automatic delete candidates.
    layer=f'dn_{uuid.uuid4().hex[:8]}'; arcpy.management.MakeFeatureLayer(str(fc),layer,sql_null(fc,'APID',False))
    groups=defaultdict(list)
    if nrows(layer)>1:
        t=SCRATCH_GDB/f'D_{uuid.uuid4().hex[:8]}'; find_identical(layer,t,['APID','Shape']); groups=identical_groups(t)
    dup_oids={oid for vals in groups.values() for oid in vals}; last_by={}
    cfields=['OID@']+([last_field.name] if last_field else [])
    if dup_oids:
        with arcpy.da.SearchCursor(str(fc),cfields) as c:
            for row in c:
                oid=int(row[0])
                if oid in dup_oids: last_by[oid]=row[1] if len(row)>1 else None
    planned=0
    with arcpy.da.InsertCursor(str(DUP_PLAN),['FC_NAME','DELETE_OID','KEEP_OID','FEAT_SEQ','DELETE_LAST_UPD','KEEP_LAST_UPD']) as c:
        for seq,oids in groups.items():
            if len(oids)<2: continue
            keep=max(oids,key=lambda oid:survivor_key(oid,last_by.get(oid))); keep_last=norm(last_by.get(keep))
            for oid in oids:
                if oid==keep: continue
                c.insertRow([fc_name,oid,keep,seq,norm(last_by.get(oid)),keep_last]); planned+=1

    # APID NULL: geometry-identical review only.
    nlayer=f'd0_{uuid.uuid4().hex[:8]}'; arcpy.management.MakeFeatureLayer(str(fc),nlayer,sql_null(fc,'APID',True))
    ngroups=defaultdict(list)
    if nrows(nlayer)>1:
        nt=SCRATCH_GDB/f'N_{uuid.uuid4().hex[:8]}'; find_identical(nlayer,nt,['Shape']); ngroups=identical_groups(nt)
    review_rows=0
    with arcpy.da.InsertCursor(str(NULL_REVIEW),['FC_NAME','FEATURE_OID','FEAT_SEQ']) as c:
        for seq,oids in ngroups.items():
            if len(oids)<2: continue
            for oid in oids: c.insertRow([fc_name,oid,seq]); review_rows+=1
    if review_rows: issue(fc_name,'DUPLICATES','NULL_APID_IDENTICAL_GEOMETRY','Identical geometry with APID=NULL retained for review.',review_rows)
    for lyr in (layer,nlayer):
        if arcpy.Exists(lyr): arcpy.management.Delete(lyr)
    return {'groups':sum(len(v)>=2 for v in groups.values()),'rows':sum(len(v) for v in groups.values() if len(v)>=2),'planned':planned,'null_groups':sum(len(v)>=2 for v in ngroups.values()),'null_rows':review_rows}

def oid_plan(table,fc_name,oid_field):
    out=set(); where=sql_eq(table,'FC_NAME',fc_name)
    with arcpy.da.SearchCursor(str(table),[oid_field],where) as c:
        for (oid,) in c:
            if oid is not None: out.add(int(oid))
    return out

def delete_oids(fc,oids):
    if not oids: return 0
    deleted=0
    with arcpy.da.UpdateCursor(str(fc),['OID@']) as c:
        for row in c:
            if int(row[0]) in oids: c.deleteRow(); deleted+=1
    return deleted

def remaining_oids(fc,oids):
    if not oids: return set()
    return {int(oid) for (oid,) in arcpy.da.SearchCursor(str(fc),['OID@']) if int(oid) in oids}

def process4_duplicates(fc_name):
    if done(fc_name,'DUPLICATES_DONE'): print('  Process 4: already complete'); return
    print('  Process 4: duplicate cleanup'); fc=PREPARE_GDB/fc_name
    if not done(fc_name,'DUP_PLAN_READY'):
        before=nrows(fc); s=build_dup_plan(fc_name)
        rec={'FC_NAME':fc_name,'COUNT_BEFORE':before,'DUP_GROUPS':s['groups'],'DUP_ROWS':s['rows'],'DUP_DELETE_PLANNED':s['planned'],'DUP_DELETED':None,'NULL_APID_IDENTICAL_GROUPS':s['null_groups'],'NULL_APID_IDENTICAL_ROWS':s['null_rows'],'COUNT_AFTER':None}
        upsert(RD,RD_SCHEMA,fc_name,rec); set_state(fc_name,DUP_PLAN_READY=1,STATUS='PROCESSING',LAST_ERROR=None)
    rec=report_row(RD,RD_SCHEMA,fc_name); planned=oid_plan(DUP_PLAN,fc_name,'DELETE_OID'); this_run=delete_oids(fc,planned); remain=remaining_oids(fc,planned)
    if remain: raise RuntimeError(f'{fc_name}: {len(remain)} duplicate planned OIDs remain')
    rec.update({'DUP_DELETED':len(planned),'COUNT_AFTER':nrows(fc)}); upsert(RD,RD_SCHEMA,fc_name,rec); set_state(fc_name,DUPLICATES_DONE=1,STATUS='PROCESSING',LAST_ERROR=None)
    print(f'    planned={len(planned):,}; deleted this run={this_run:,}')


## Process 5 — CHANGWAT / 2 km validation

The correct reference feature class is `CHANGWAT_47` or `CHANGWAT_48`. Temporary point features are constructed from the exact stored `INSIDE_X/Y` coordinates, carrying the source OID. Points farther than 2,000 metres from the selected province polygon are planned for deletion.

In [7]:
def make_inside_points(fc,out_fc):
    if arcpy.Exists(str(out_fc)): arcpy.management.Delete(str(out_fc))
    sr=arcpy.Describe(str(fc)).spatialReference
    arcpy.management.CreateFeatureclass(str(SCRATCH_GDB),Path(out_fc).name,'POINT',spatial_reference=sr)
    arcpy.management.AddField(str(out_fc),'SRC_OID','LONG')
    inserted=nulls=0
    with arcpy.da.SearchCursor(str(fc),['OID@','INSIDE_X','INSIDE_Y']) as src, arcpy.da.InsertCursor(str(out_fc),['SHAPE@XY','SRC_OID']) as dst:
        for oid,x,y in src:
            if x is None or y is None: nulls+=1; continue
            dst.insertRow([(float(x),float(y)),int(oid)]); inserted+=1
            if POINT_PROGRESS_EVERY and inserted%POINT_PROGRESS_EVERY==0: print(f'    built {inserted:,} inside points')
    return inserted,nulls

def reference_layer(ident):
    ref_name=f"CHANGWAT_{ident['ZONE']}"; ref=REFERENCE_GDB/ref_name
    if not arcpy.Exists(str(ref)): return None,ref_name,0,f'{ref_name} does not exist'
    if wkid(ref)!=ZONE_WKID[ident['ZONE']]: return None,ref_name,0,f'{ref_name}: WKID {wkid(ref)}, expected {ZONE_WKID[ident["ZONE"]]}'
    if not field(ref,'CHANGWAT_CODE'): return None,ref_name,0,f'{ref_name}: CHANGWAT_CODE field missing'
    lyr=f'ref_{uuid.uuid4().hex[:8]}'; arcpy.management.MakeFeatureLayer(str(ref),lyr,sql_eq(ref,'CHANGWAT_CODE',ident['CHANGWAT_CODE']))
    cnt=nrows(lyr)
    if cnt==0:
        arcpy.management.Delete(lyr); return None,ref_name,0,f"CHANGWAT_CODE={ident['CHANGWAT_CODE']} not found in {ref_name}"
    return lyr,ref_name,cnt,None

def build_boundary_plan(fc_name,ident):
    fc=PREPARE_GDB/fc_name; delete_fc_rows(BOUNDARY_PLAN,fc_name); clear_issues(fc_name,'BOUNDARY'); before=nrows(fc)
    ref,ref_name,ref_count,reason=reference_layer(ident)
    if ref is None:
        issue(fc_name,'BOUNDARY','BOUNDARY_REFERENCE_SKIPPED',reason,1)
        return {'skipped':True,'ref':ref_name,'ref_count':ref_count,'before':before,'points':0,'nulls':0,'keep':0,'delete':0,'reason':reason}
    pts=SCRATCH_GDB/f'P_{uuid.uuid4().hex[:8]}'
    try:
        point_count,nulls=make_inside_points(fc,pts); lyr=f'pt_{uuid.uuid4().hex[:8]}'; arcpy.management.MakeFeatureLayer(str(pts),lyr)
        arcpy.management.SelectLayerByLocation(lyr,'WITHIN_A_DISTANCE',ref,BOUNDARY_DISTANCE,'NEW_SELECTION')
        keep=nrows(lyr); arcpy.management.SelectLayerByAttribute(lyr,'SWITCH_SELECTION'); delete_count=nrows(lyr)
        with arcpy.da.InsertCursor(str(BOUNDARY_PLAN),['FC_NAME','DELETE_OID']) as ins, arcpy.da.SearchCursor(lyr,['SRC_OID']) as src:
            for (oid,) in src: ins.insertRow([fc_name,int(oid)])
        if nulls: issue(fc_name,'BOUNDARY','INSIDE_XY_NULL_NOT_DELETED','Rows with NULL INSIDE_X/Y retained because boundary test is impossible.',nulls)
        if arcpy.Exists(lyr): arcpy.management.Delete(lyr)
        return {'skipped':False,'ref':ref_name,'ref_count':ref_count,'before':before,'points':point_count,'nulls':nulls,'keep':keep,'delete':delete_count,'reason':None}
    finally:
        if arcpy.Exists(ref): arcpy.management.Delete(ref)
        if arcpy.Exists(str(pts)): arcpy.management.Delete(str(pts))

def process5_boundary(fc_name,ident):
    if done(fc_name,'BOUNDARY_DONE'): print('  Process 5: already complete'); return
    print('  Process 5: CHANGWAT / 2 km INSIDE_X/Y'); fc=PREPARE_GDB/fc_name
    if not done(fc_name,'BOUNDARY_PLAN_READY'):
        s=build_boundary_plan(fc_name,ident)
        rec={'FC_NAME':fc_name,'CHANGWAT_CODE':ident['CHANGWAT_CODE'],'ZONE':ident['ZONE'],'REFERENCE_FC':s['ref'],'REFERENCE_MATCH_COUNT':s['ref_count'],'COUNT_BEFORE':s['before'],'INSIDE_POINT_COUNT':s['points'],'INSIDE_POINT_NULL':s['nulls'],'KEEP_WITHIN_2KM':s['keep'],'DELETE_PLANNED':s['delete'],'DELETE_COUNT':None,'COUNT_AFTER':None,'STATUS':'SKIPPED' if s['skipped'] else 'PLANNED','SKIP_REASON':s['reason']}
        upsert(RB,RB_SCHEMA,fc_name,rec); set_state(fc_name,BOUNDARY_PLAN_READY=1,STATUS='PROCESSING',LAST_ERROR=None)
        if s['skipped']:
            rec.update({'DELETE_COUNT':0,'COUNT_AFTER':nrows(fc),'STATUS':'SKIPPED'}); upsert(RB,RB_SCHEMA,fc_name,rec); set_state(fc_name,BOUNDARY_DONE=1,STATUS='PROCESSING',LAST_ERROR=None); return
    rec=report_row(RB,RB_SCHEMA,fc_name)
    if rec['STATUS']=='SKIPPED': set_state(fc_name,BOUNDARY_DONE=1,STATUS='PROCESSING',LAST_ERROR=None); return
    planned=oid_plan(BOUNDARY_PLAN,fc_name,'DELETE_OID'); this_run=delete_oids(fc,planned); remain=remaining_oids(fc,planned)
    if remain: raise RuntimeError(f'{fc_name}: {len(remain)} boundary planned OIDs remain')
    rec.update({'DELETE_COUNT':len(planned),'COUNT_AFTER':nrows(fc),'STATUS':'COMPLETE'}); upsert(RB,RB_SCHEMA,fc_name,rec); set_state(fc_name,BOUNDARY_DONE=1,STATUS='PROCESSING',LAST_ERROR=None)
    print(f'    planned={len(planned):,}; deleted this run={this_run:,}')


## Reporting and reconciliation

The Excel workbook is regenerated from persistent GDB report tables. If the workbook is open/locked, processing continues and the newly generated temporary workbook is left in the report directory.

In [8]:
def report_value(table,schema,fc,name,default=0):
    r=report_row(table,schema,fc)
    if not r or r.get(name) is None: return default
    return r[name]

def update_fc_report(fc_name,ident,status):
    s=state(fc_name) or {}; p=report_row(R1,R1_SCHEMA,fc_name) or {}; g=report_row(RG,RG_SCHEMA,fc_name) or {}; d=report_row(RD,RD_SCHEMA,fc_name) or {}; b=report_row(RB,RB_SCHEMA,fc_name) or {}
    rec={'FC_NAME':fc_name,'BRANCH_CODE':ident['BRANCH_CODE'],'CHANGWAT_CODE':ident['CHANGWAT_CODE'],'ZONE':ident['ZONE'],'WKID':ident['WKID'],'COUNT_START':s.get('START_COUNT'),'APID_NULL':p.get('APID_NULL'),'GEOM_PROBLEM_FEATURES':g.get('PROBLEM_FEATURES_BEFORE'),'DUP_DELETED':d.get('DUP_DELETED'),'BOUNDARY_DELETED':b.get('DELETE_COUNT'),'COUNT_FINAL':nrows(PREPARE_GDB/fc_name),'STATUS':status}
    upsert(RFC,RFC_SCHEMA,fc_name,rec)

def sum_field(table,name):
    return sum(int(v or 0) for (v,) in arcpy.da.SearchCursor(str(table),[name]))

def rebuild_summary(discovered):
    delete_where(RS); statuses=Counter(v for (v,) in arcpy.da.SearchCursor(str(STATE),['STATUS'])); skipped=sum(1 for (v,) in arcpy.da.SearchCursor(str(RB),['STATUS']) if v=='SKIPPED')
    metrics={'FEATURE_CLASSES_DISCOVERED':discovered,'FEATURE_CLASSES_COMPLETE':statuses['COMPLETE'],'FEATURE_CLASSES_ERROR':statuses['ERROR'],'TOTAL_COUNT_START':sum_field(RFC,'COUNT_START'),'TOTAL_COUNT_FINAL':sum_field(RFC,'COUNT_FINAL'),'TOTAL_APID_NULL':sum_field(R1,'APID_NULL'),'TOTAL_GEOMETRY_PROBLEM_FEATURES_BEFORE':sum_field(RG,'PROBLEM_FEATURES_BEFORE'),'TOTAL_GEOMETRY_PROBLEM_FEATURES_AFTER':sum_field(RG,'PROBLEM_FEATURES_AFTER'),'TOTAL_DUPLICATE_DELETED':sum_field(RD,'DUP_DELETED'),'TOTAL_NULL_APID_IDENTICAL_REVIEW_ROWS':sum_field(RD,'NULL_APID_IDENTICAL_ROWS'),'TOTAL_BOUNDARY_DELETED':sum_field(RB,'DELETE_COUNT'),'BOUNDARY_SKIPPED_FEATURE_CLASSES':skipped}
    with arcpy.da.InsertCursor(str(RS),['METRIC','VALUE']) as c:
        for k,v in metrics.items(): c.insertRow([k,int(v or 0)])

def write_report(discovered):
    rebuild_summary(discovered); tables=[RS,R1,RG,RD,RB,RFC,ISSUES]; tmp=REPORT_DIR/f'_tmp_{ACQUIRED.replace("-","_")}_{uuid.uuid4().hex[:8]}.xlsx'
    try:
        arcpy.conversion.TableToExcel([str(t) for t in tables],str(tmp))
        try:
            if ACQUIRED_REPORT.exists(): ACQUIRED_REPORT.unlink()
            os.replace(tmp,ACQUIRED_REPORT); print('  Report updated:',ACQUIRED_REPORT)
        except PermissionError:
            print('  WARNING: report is open/locked; new workbook kept at',tmp)
    except Exception as exc: print('  WARNING: Excel report failed; control-GDB report data is safe:',exc)

def reconcile(fc_name):
    s=state(fc_name); start=int(s['START_COUNT']); dup=int(report_value(RD,RD_SCHEMA,fc_name,'DUP_DELETED',0)); bnd=int(report_value(RB,RB_SCHEMA,fc_name,'DELETE_COUNT',0)); final=nrows(PREPARE_GDB/fc_name); expected=start-dup-bnd
    if expected!=final:
        issue(fc_name,'FINAL','COUNT_RECONCILIATION_FAILED',f'start={start}, duplicate_deleted={dup}, boundary_deleted={bnd}, expected={expected}, actual={final}',abs(expected-final) or 1)
        raise RuntimeError(f'{fc_name}: final count reconciliation failed')
    return final

def clear_scratch_contents():
    old=arcpy.env.workspace
    try:
        arcpy.env.workspace=str(SCRATCH_GDB)
        for name in (arcpy.ListFeatureClasses() or [])+(arcpy.ListTables() or []):
            try: arcpy.management.Delete(str(SCRATCH_GDB/name))
            except: pass
    finally: arcpy.env.workspace=old


## Run

Normal rerun is safe: completed feature classes are skipped; incomplete stage flags and persisted delete plans are reused.

In [9]:
def run_first_clean():
    setup(); ensure_control(); fcs=list_fcs(); print(f'Discovered {len(fcs):,} matching feature class(es).')
    completed=skipped=failed=0
    for i,(fc_name,branch,zone) in enumerate(fcs,1):
        ident=None; print(f'\n[{i}/{len(fcs)}] {fc_name}')
        try:
            ident=identity(fc_name,branch,zone); fc=PREPARE_GDB/fc_name; current=nrows(fc); s=ensure_state(fc_name,current)
            if s['STATUS']=='COMPLETE':
                if s.get('FINAL_COUNT') is not None and int(s['FINAL_COUNT'])!=current: raise RuntimeError(f'{fc_name}: COMPLETE count changed from {s["FINAL_COUNT"]} to {current}; external edit suspected')
                skipped+=1; print('  COMPLETE checkpoint — skip'); update_fc_report(fc_name,ident,'COMPLETE'); continue
            set_state(fc_name,STATUS='PROCESSING',LAST_ERROR=None)
            process1(fc_name,ident); process2_geometry(fc_name); process3_inside(fc_name); process4_duplicates(fc_name); process5_boundary(fc_name,ident)
            final=reconcile(fc_name); set_state(fc_name,STATUS='COMPLETE',FINAL_COUNT=final,LAST_ERROR=None); update_fc_report(fc_name,ident,'COMPLETE'); completed+=1; print(f'  COMPLETE | final={final:,}')
        except KeyboardInterrupt:
            if state(fc_name): set_state(fc_name,STATUS='INTERRUPTED',LAST_ERROR='User interrupted; rerun notebook to resume.')
            issue(fc_name,'SYSTEM','INTERRUPTED','User interrupted; rerun notebook to resume.',1)
            if ident: update_fc_report(fc_name,ident,'INTERRUPTED')
            if WRITE_REPORT_AFTER_EACH_FC: write_report(len(fcs))
            print('Execution interrupted. Persistent checkpoints were kept.'); raise
        except Exception as exc:
            failed+=1; msg=f'{type(exc).__name__}: {exc}'; print('  ERROR:',msg); traceback.print_exc()
            if state(fc_name): set_state(fc_name,STATUS='ERROR',LAST_ERROR=msg)
            issue(fc_name,'SYSTEM','PROCESSING_ERROR',msg,1)
            if ident: update_fc_report(fc_name,ident,'ERROR')
            if STOP_ON_ERROR: raise
        finally:
            clear_scratch_contents()
            if WRITE_REPORT_AFTER_EACH_FC: write_report(len(fcs))
    write_report(len(fcs))
    if not KEEP_SCRATCH_GDB and arcpy.Exists(str(SCRATCH_GDB)): arcpy.management.Delete(str(SCRATCH_GDB))
    print('\nFIRST CLEAN PASS FINISHED')
    print('Feature classes   :',len(fcs)); print('Completed this run:',completed); print('Skipped COMPLETE  :',skipped); print('Failed this run   :',failed); print('Control GDB       :',CONTROL_GDB); print('Excel report      :',ACQUIRED_REPORT)
    return {'FEATURE_CLASSES':len(fcs),'COMPLETED_THIS_RUN':completed,'SKIPPED_COMPLETE':skipped,'FAILED_THIS_RUN':failed,'CONTROL_GDB':str(CONTROL_GDB),'ACQUIRED_REPORT':str(ACQUIRED_REPORT)}

RESULT = run_first_clean()
RESULT


Discovered 81 matching feature class(es).

[1/81] A10080000_47
  Process 1: APID + ACQUIRED
  Process 2: Check + Repair Geometry
  Process 3: INSIDE_X + INSIDE_Y
  Process 4: duplicate cleanup
    planned=5; deleted this run=5
  Process 5: CHANGWAT / 2 km INSIDE_X/Y
    planned=8; deleted this run=8
  COMPLETE | final=83,094
  Report updated: D:\A02-Projects\Refine\05_reports\2022-05\02_2022-05-first_Clean.xlsx

[2/81] A10090000_47
  Process 1: APID + ACQUIRED
  Process 2: Check + Repair Geometry
  Process 3: INSIDE_X + INSIDE_Y
  Process 4: duplicate cleanup
    planned=101; deleted this run=101
  Process 5: CHANGWAT / 2 km INSIDE_X/Y
    planned=0; deleted this run=0
  COMPLETE | final=204,589
  Report updated: D:\A02-Projects\Refine\05_reports\2022-05\02_2022-05-first_Clean.xlsx

[3/81] A10100000_47
  Process 1: APID + ACQUIRED
  Process 2: Check + Repair Geometry
  Process 3: INSIDE_X + INSIDE_Y
  Process 4: duplicate cleanup
    planned=41; deleted this run=41
  Process 5: CHANGWA

{'FEATURE_CLASSES': 81,
 'COMPLETED_THIS_RUN': 81,
 'SKIPPED_COMPLETE': 0,
 'FAILED_THIS_RUN': 0,
 'CONTROL_GDB': 'D:\\A02-Projects\\Refine\\05_reports\\2022-05\\_clean_control_2022_05.gdb',
 'ACQUIRED_REPORT': 'D:\\A02-Projects\\Refine\\05_reports\\2022-05\\02_2022-05-first_Clean.xlsx'}

## Resume / restart notes

### Normal resume

Close the Excel report if it is open, then run this notebook from the top.

- `COMPLETE` feature classes are skipped.
- APID, geometry repair, and INSIDE coordinate stages are idempotent and rerun only if their checkpoint was not completed.
- Duplicate and boundary deletion plans are persistent. If deletion was interrupted, the same planned OIDs are reused.
- Detailed `APID=NULL` identical-geometry review rows remain in `NULL_APID_REVIEW` inside the control GDB rather than being exported to a potentially oversized Excel sheet.

### Full restart

Because this notebook performs destructive deletions, deleting only the control GDB is **not** a valid full restart after cleaning has begun.

A true restart requires:

1. rebuild/restore `PREPARE_GDB` from the import stage;
2. delete `_clean_control_<ACQUIRED>.gdb`;
3. run this notebook again.

### Validation status

The notebook's Python cells have been syntax-validated and the notebook structure is validated with `nbformat`. It cannot be executed here because ArcPy and the local `D:\...` geodatabases are not available in this environment.